In [ ]:
!uv add -r rag-requirements.txt

In [6]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a vigent city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[get_weather],
    system_prompt = "You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in the Tokyo?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'It’s sunny in Tokyo.'}]


In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results = 2)
tool.invoke({"query": "도쿄의 날씨는 어떤가요?"})

In [8]:
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.tools import tool
load_dotenv()

class Answer(BaseModel):
    summary: str
    confidence: float

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


agent = create_agent(
    model="openai:gpt-5.5",
    tools=[search],
    response_format=Answer,
    system_prompt="너는 학습 도우미야, 핵심과 간단구조만 알려줘",
)

result = agent.invoke({"messages": [{"role": "user", "content": "싱글 에이전트를 설명해줘"}]})

answer = result["structured_response"]

print(answer.summary)
print(f"\nconfidence: {answer.confidence}")

싱글 에이전트(Single Agent)는 하나의 AI 에이전트가 목표를 이해하고, 계획을 세우고, 도구를 사용하며, 결과를 만들어내는 구조입니다.

핵심 구조:
1. 입력: 사용자의 요청을 받음
2. 판단: 무엇을 해야 하는지 분석
3. 실행: 필요한 작업 수행 또는 도구 사용
4. 출력: 결과를 사용자에게 제공

예시:
- 사용자가 “오늘 일정 정리해줘”라고 하면
- 하나의 에이전트가 일정을 확인하고 정리해서 답변함

장점:
- 구조가 단순함
- 만들고 관리하기 쉬움
- 비용과 복잡도가 낮음

단점:
- 복잡한 문제를 여러 전문 역할로 나누기 어려움
- 한 에이전트가 모든 일을 처리해야 해서 한계가 있음

비교:
- 싱글 에이전트: 혼자 일하는 AI
- 멀티 에이전트: 여러 AI가 역할을 나눠 협업하는 구조

confidence: 0.97


In [2]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Adds a and b.
    
    Args:
        a: first int
        b: second int
    """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b.
    Args:
        a: first int
        b: second int
    """
    return a * b
tools = [add, multiply]

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4o")
llm_with_tools = llm.bind_tools(tools)

query = "19587 곱하기 39574는 뭔가요? 그리고 193847535487 더하기 193828457573는 뭔가요?"

response = llm_with_tools.invoke(query)
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 128, 'total_tokens': 186, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_70aa11d053', 'id': 'chatcmpl-EVpBbl106hiAwFfpVcaB8Q5tlTSCs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10f09-5f86-7c72-8bea-b43f1e51f589-0', tool_calls=[{'name': 'multiply', 'args': {'a': 19587, 'b': 39574}, 'id': 'call_fgIdnvPGQjYNs1I0LwSbCsmC', 'type': 'tool_call'}, {'name': 'add', 'args': {'a': 193847535487, 'b': 193828457573}, 'id': 'call_JLB2zRxxjXY1fQQ5zQvfd2rH', 'type': 'tool_call'}], invali

In [10]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 19587, 'b': 39574},
  'id': 'call_fgIdnvPGQjYNs1I0LwSbCsmC',
  'type': 'tool_call'},
 {'name': 'add',
  'args': {'a': 193847535487, 'b': 193828457573},
  'id': 'call_JLB2zRxxjXY1fQQ5zQvfd2rH',
  'type': 'tool_call'}]

In [11]:
query = "안녕하세요"

response = llm_with_tools.invoke(query)
response

AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 97, 'total_tokens': 109, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_70aa11d053', 'id': 'chatcmpl-EVpD2OkvQuNICzSeQzt0cogFllhRT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a10f0a-ba44-7e02-ac1d-af52fbcb33db-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 12, 'total_tokens': 109, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [12]:
response.tool_calls

[]

In [1]:
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=3)
tools = [tool]

llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

In [2]:
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

In [3]:
def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}
graph_builder.add_node("chatbot", chatbot)

In [4]:
import json
from langchain.messages import ToolMessage

class BasicToolNode:
    """
    마지막 AIMessage에서 요청된 도구를 실행하는 노트
    """
    def __init__(self, tools: list) -> None:
        self.tools_by_name = {tool.name: tool for tool in tools}

    def __call__(self, inputs: dict):
        if messages := inputs.get("messages", []):
            message = messages[-1]
        else:
            raise ValueError("ERROR: 입력에 메시지가 없습니다.")
        outputs = []
        for tool_call in message.tool_calls:
            tool_result = self.tools_by_name[tool_call["name"]].invoke(
                tool_call["args"]
            )
            outputs.append(
                ToolMessage(
                    content=json.dumps(tool_result, ensure_ascii=False),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )
        return {"messages": outputs}
tool_node = BasicToolNode(tools = [tool])
graph_builder.add_node("tools", tool_node)


In [5]:
def route_tools(
        state: State,
):
    """
    마지막 메시지에 도구 호출이 있는 경우, ToolNode로 라우팅하고 그렇지 않으면 END로 라우팅
    """
    if isinstance(state,list):
        ai_message = state[-1]
    elif messages := state.get("messages",[]):
        ai_message = messages[-1]
    else:
        raise ValueError(f"ERROR: 입력에 메시지가 없습니다. 상태: {state}")
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) >0:
        return "tools"
    return END

graph_builder.add_conditional_edges(
    "chatbot",
    route_tools,
    {"tools": "tools", END: END},
)

In [6]:
graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge(START, "chatbot")
graph = graph_builder.compile()

In [7]:
if __name__ == "__main__":
    try:
        image= graph.get_graph().draw_mermaid_png()
        with open("web_agent/graph.png", "wb")as f:
            f.write(image)
    except Exception:
        pass
    response = graph.invoke(
        {
            "messages": ["도쿄의 날씨는 어떤가요?"]
        }
    )

    print(response)

{'messages': [HumanMessage(content='도쿄의 날씨는 어떤가요?', additional_kwargs={}, response_metadata={}, id='34c36f22-4625-4c99-bc5f-70bf938c1f23'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1198, 'total_tokens': 1221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1024, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_0ae67b3da2', 'id': 'chatcmpl-EVqFcHciNyBVCpivKPlor62np381W', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10f47-d0c3-76d3-aa75-2e34f35ac7d8-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in Tokyo', 'topic': 'news'}, 'id': 'call_R

In [8]:
def invoke():
    response = graph.invoke(
        {
            "messages": ["도쿄의 날씨는 어떤가요?"]
        }
    )

    for msg in response["messages"]:
        msg.pretty_print()

if __name__ == "__main__":
    invoke()

================================ Human Message =================================

도쿄의 날씨는 어떤가요?
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_fuujOfqKwXfiIGpuuZCouuTS)
 Call ID: call_fuujOfqKwXfiIGpuuZCouuTS
  Args:
    query: Tokyo weather
    topic: news
    time_range: day
================================= Tool Message =================================
Name: tavily_search

{"query": "Tokyo weather", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.cbsnews.com/boston/video/medford-high-school-evacuated-due-to-sewage-backup/", "title": "Medford High School evacuated due to sewage backup - CBS News", "score": 0.17439653, "published_date": "Mon, 05 Oct 2026 21:53:00 GMT", "content": "# Medford High School evacuated due to sewage backup. Firefighters ordered Medford High School to be closed and evacuated due to a sewage backup. ### Next Weather: WBZ evening forecast for October

In [9]:
async def ainvoke():
    response = await graph.ainvoke(
        {
            "messages": ["도쿄의 날씨는 어떤가요?"]
        }
    )

    for msg in response["messages"]:
        msg.pretty_print()

if __name__ == "__main__":
    import asyncio
    asyncio.run(ainvoke())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
!uv add -U langgraph-cli[inmem]